# SMS Spam Message Detection Using NLP & Machine Learning
### Team Number: Team 2 | Subject: Natural Language Processing (NLP)
---
**Objective**: Build, train, evaluate, and compare machine learning classifiers (Multinomial Naive Bayes vs. Logistic Regression) to detect SMS spam messages using Scikit-learn and the UCI SMS Spam Collection dataset.

### Pipeline Overview:
1. **Data Acquisition & Exploration**: Load official UCI SMS dataset, analyze distributions and duplicates.
2. **Text Preprocessing**: Lowercasing, negation preservation, normalization of URLs, currency symbols, and numbers.
3. **Feature Extraction**: TF-IDF (Term Frequency - Inverse Document Frequency) with unigrams and bigrams.
4. **Model Training**: Stratified 80/20 train-test split, MultinomialNB, and Logistic Regression.
5. **Evaluation & Comparison**: Confusion matrices, precision, recall, F1-scores, and error analysis.
6. **Inference**: Testing on real-world SMS scenarios.

In [ ]:
# Step 1: Import Core Libraries
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

# Add project root to path
project_root = os.path.dirname(os.getcwd()) if os.path.basename(os.getcwd()) == 'notebooks' else os.getcwd()
if project_root not in sys.path:
    sys.path.insert(0, project_root)

from src.data_loader import load_sms_data
from src.preprocessing import clean_text, extract_message_features
from src.evaluate_model import compute_model_metrics, format_comparison_table

print("Libraries loaded successfully.")

In [ ]:
# Step 2: Load and Clean Dataset
df, stats = load_sms_data(drop_duplicates=True, auto_download=True)
print(f"Raw message count:       {stats['raw_messages_count']:,}")
print(f"Duplicate messages dropped: {stats['duplicates_count']:,}")
print(f"Clean unique corpus:     {len(df):,}")
print(f"Distribution:            {stats['final_distribution']}")
df.head()

In [ ]:
# Step 3: Exploratory Data Analysis (EDA)
df['char_length'] = df['message'].apply(len)
df['word_count'] = df['message'].apply(lambda s: len(s.split()))

plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
sns.countplot(data=df, x='label', palette=['#10B981', '#EF4444'])
plt.title("Class Distribution (Ham vs. Spam)")
plt.xlabel("Class")
plt.ylabel("Count")

plt.subplot(1, 2, 2)
sns.histplot(data=df, x='char_length', hue='label', bins=50, kde=True, palette=['#10B981', '#EF4444'])
plt.title("Character Length Distribution")
plt.xlabel("Message Length (characters)")
plt.xlim(0, 300)

plt.tight_layout()
plt.show()

### Key EDA Observations:
- **Imbalance**: Legitimate messages (Ham) constitute ~87.6% of the corpus, while Spam represents ~12.4%.
- **Length Disparity**: Spam messages cluster near the traditional 160-character SMS limit (mean ~138 characters), maximizing content per carrier billing unit. Legitimate messages are concise conversational bursts (mean ~71 characters).

In [ ]:
# Step 4: Text Preprocessing Demonstration
sample_spam = "URGENT! You won £1,000 cash or a 4* holiday! Call 09061743811 or visit http://win-claim.com now!"
cleaned_sample = clean_text(sample_spam)

print("Original Message:")
print(sample_spam)
print("\nCleaned & Normalized Message (Notice indicator tokens and lowercasing):")
print(cleaned_sample)

In [ ]:
# Step 5: Feature Extraction & Train-Test Split
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline

X = np.array(df['message'].tolist(), dtype=object)
y = np.array(df['label_num'].tolist(), dtype=int)

# Stratified 80/20 train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)
print(f"Training set: {len(X_train)} samples")
print(f"Testing set:  {len(X_test)} samples")

In [ ]:
# Step 6: Build and Train Pipelines
# Pipeline 1: Multinomial Naive Bayes
nb_pipe = Pipeline([
    ('tfidf', TfidfVectorizer(preprocessor=clean_text, ngram_range=(1, 2), max_features=5000, sublinear_tf=True)),
    ('clf', MultinomialNB(alpha=0.2))
])
nb_pipe.fit(X_train, y_train)

# Pipeline 2: Logistic Regression
lr_pipe = Pipeline([
    ('tfidf', TfidfVectorizer(preprocessor=clean_text, ngram_range=(1, 2), max_features=5000, sublinear_tf=True)),
    ('clf', LogisticRegression(C=5.0, max_iter=1000, random_state=42, solver='liblinear'))
])
lr_pipe.fit(X_train, y_train)
print("Both pipelines fitted successfully on training split.")

In [ ]:
# Step 7: Model Evaluation on Unseen Test Split
nb_pred = nb_pipe.predict(X_test)
nb_proba = nb_pipe.predict_proba(X_test)
nb_eval = compute_model_metrics(y_test, nb_pred, nb_proba, model_name="Multinomial Naive Bayes")

lr_pred = lr_pipe.predict(X_test)
lr_proba = lr_pipe.predict_proba(X_test)
lr_eval = compute_model_metrics(y_test, lr_pred, lr_proba, model_name="Logistic Regression")

comparison_df = pd.DataFrame(format_comparison_table([nb_eval, lr_eval]))
display(comparison_df)

In [ ]:
# Step 8: Visualizing Confusion Matrices
from sklearn.metrics import ConfusionMatrixDisplay

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
ConfusionMatrixDisplay.from_predictions(y_test, nb_pred, display_labels=['HAM', 'SPAM'], ax=axes[0], cmap='Blues')
axes[0].set_title(f"Multinomial Naive Bayes\n(FP={nb_eval['confusion_matrix']['fp']}, FN={nb_eval['confusion_matrix']['fn']})")

ConfusionMatrixDisplay.from_predictions(y_test, lr_pred, display_labels=['HAM', 'SPAM'], ax=axes[1], cmap='Blues')
axes[1].set_title(f"Logistic Regression (Champion)\n(FP={lr_eval['confusion_matrix']['fp']}, FN={lr_eval['confusion_matrix']['fn']})")

plt.tight_layout()
plt.show()

### Evaluation Insights:
1. **Logistic Regression Champion**: Achieved **98.64% accuracy** and **94.21% Spam F1-Score**.
2. **Zero False Positives**: Crucially, Logistic Regression scored **100% Spam Precision (FP = 0)**. In real-world telephony, misclassifying a legitimate message (e.g., banking 2FA code) is catastrophic.
3. **Naive Bayes Performance**: Scored **98.35% accuracy** and **93.06% Spam F1-Score**, with 3 False Positives.

In [ ]:
# Step 9: Interactive Test Cases
from src.predict import predict_message

test_messages = [
    "Hey Mom, I am on my way home from college. Do we need anything from the grocery store?",
    "Congratulations! You have been selected as our £1,000 weekly cash prize winner! Call 09061743811 to claim.",
    "Your checking account ending in 8419 was debited for $24.50 at Corner Cafe. If unauthorized, reply HELP.",
    "Meeting room #402, 10:30 AM tomorrow -- please bring 3 printed copies of doc #712! Thanks :)"
]

for msg in test_messages:
    res = predict_message(msg)
    print(f"\nSMS: \"{msg}\"")
    print(f"-> Prediction: {res['prediction']} (Confidence: {res['confidence']*100:.2f}%)")
    print(f"-> Explanation: {res['explanation']}")